# Getting Started with Firewall Logs Agent Development

Welcome to the Firewall Automation Hackathon! This notebook will guide you through building the Firewall Logs Agent as a standalone agent.

## Overview

The Firewall Logs Agent is a **standalone specialist agent** that handles all firewall log analysis. It will be invoked by the supervisor agent when log queries are needed. This agent:
- Queries AWS Network Firewall logs from OpenSearch Serverless
- Translates natural language queries to OpenSearch DSL
- Parses and summarizes log results
- Identifies patterns, anomalies, and security issues

## Example Implementation

A complete working example is provided in `network_firewall_analyser_agent.py`. This demonstrates:
- Cross-account role assumption for OpenSearch access
- Tool definitions using the `@tool` decorator
- OpenSearch query building and execution
- Result parsing and formatting
- Full agent implementation with async streaming

## How to Get Started

Review the base agent code and the example implementation. The example uses [Strands Agents SDK](https://strandsagents.com/latest/) to implement tools that convert natural language into OpenSearch queries. 

Your task is to create a **new standalone agent** based on the example, then deploy it separately so the supervisor agent can invoke it.

### Authentication Setup

In the example, we use IAM to establish cross-account access to OpenSearch deployed in the networking account:

```
IAM Role for Sagemaker -> IAM Role for OpenSearch Access (in this account) ---cross-account---> OpenSearch
```

The IAM Role for OpenSearch Access is: `arn:aws:iam::123456789012:role/YourOpenSearchAccessRole`

### Alternative Approach: MCP Server

After implementing the direct OpenSearch integration, explore the MCP server for OpenSearch:
https://docs.aws.amazon.com/opensearch-service/latest/developerguide/cfn-template-mcp-server.html

Once the example implementation is working, try the MCP server implementation and compare the differences.

## Related Jira Task

- [FWAUTO-13: Firewall logs agent implemented (Phase 1)](https://your-jira-instance.atlassian.net/browse/FWAUTO-13)

## Step 1: Copy the Base Agent Code

Copy the supervisor agent code to your workspace so you can modify it.

In [ ]:
%%bash
# Copy the base agent code from the repository
cp -r /home/sagemaker-user/IST-AWS-Firewall-Automation/agent .

In [ ]:
# View the base agent python code
with open('agent/src/agent.py', 'r') as f:
    print(f.read())

## Step 2: Review the Example Implementation

Study the complete example to understand the patterns.

In [ ]:
# View the example implementation
with open('network_firewall_analyser_agent.py', 'r') as f:
    print(f.read())

## Step 3: Create Your Firewall Logs Agent

Create a new standalone agent based on the example implementation. This agent will run separately and be invoked by the supervisor agent.

### Key Implementation Steps:

1. **Create a new agent file** - Start with `firewall_logs_agent.py` based on the example
2. **Set up cross-account access** - Use STS AssumeRole to access OpenSearch in another account
3. **Create OpenSearch client** - Use boto3 with SigV4 authentication
4. **Implement agent tools** - Define tools for searching, analyzing, and summarizing logs:
   - `search_firewall_logs()` - Flexible log searching with filters
   - `get_blocked_traffic_by_account()` - Security-focused analysis
   - `get_top_urls_by_account()` - Traffic pattern analysis
   - `get_traffic_summary_by_account()` - Comprehensive summaries
5. **Add error handling** - Handle connection failures, timeouts, and invalid queries
6. **Implement AgentCore entrypoint** - Set up the async streaming entrypoint

### Implementation Options:

#### Option 1: Direct Implementation (Recommended for Learning)
Adapt the code from `network_firewall_analyser_agent.py`:
- Copy the complete agent structure
- Keep the OpenSearch connection logic
- Keep the tool definitions
- Adapt for AgentCore Runtime deployment
- Add AgentCore entrypoint wrapper

This is a **specialist agent** focused solely on firewall log analysis:
- It doesn't need to coordinate other agents
- It has deep expertise in OpenSearch queries and log analysis
- It provides detailed, structured responses about firewall traffic
- The supervisor agent will invoke this agent when users ask about logs

#### Option 2: Use MCP Server
Once option 1 is finished, move on to option 2 and observe the difference in implementation. The MCP server approach provides a simpler integration with standardized interfaces.

In [ ]:
# View the current dummy implementation
with open('agent/src/agent.py', 'r') as f:
    content = f.read()
    # Find and display the search_firewall_logs function
    start = content.find('def search_firewall_logs')
    end = content.find('\n\n@tool', start)
    if start != -1:
        print(content[start:end if end != -1 else start+1000])

## Step 4: Test Your Implementation

Test the tool locally before deploying.

In [ ]:
# TODO: Add your test code here
# Example test:
from agent.src.agent import search_firewall_logs
result = search_firewall_logs(
    query="10.100.50.10",
    time_range_hours=1,
    account_filter="123456789012"
)
print(result)

## Step 5: Deploy to AWS

Deploy your updated agent to AWS Bedrock AgentCore Runtime.

In [ ]:
import time
import boto3
from bedrock_agentcore_starter_toolkit import Runtime

# Initialize the runtime toolkit
region = "ap-southeast-2"

agentcore_runtime = Runtime()

# Configure the deployment
response = agentcore_runtime.configure(
    agent_name=<AGENT_NAME>,  # TODO: Set your agent name, e.g., "firewall-logs-agent"
    entrypoint=<ENTRYPOINT>,  # TODO: Set your entrypoint file, e.g., "firewall_logs_agent.py"
    execution_role="arn:aws:iam::123456789012:role/YourAgentCoreExecutionRole",
    code_build_execution_role="arn:aws:iam::123456789012:role/YourCodeBuildRole",
    auto_create_ecr=True,
    requirements_file=<REQUIREMENTS_FILE>,  # TODO: Set your requirements file, e.g., "requirements.txt"
    region=region,
    memory_mode="STM_ONLY",
)

print("Configuration completed:", response)

launch_result = agentcore_runtime.launch()
print("Launch completed:", launch_result.agent_arn)

# Wait for the agent to be ready
status_response = agentcore_runtime.status()
status = status_response.endpoint["status"]

end_status = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]
while status not in end_status:
    print(f"Waiting for deployment... Current status: {status}")
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint["status"]

if status == "READY":
    runtime_id = status_response.agent["agentRuntimeId"]

    # Update the runtime to be deployed in VPC
    client = boto3.client("bedrock-agentcore-control", region_name=region)

    response = client.update_agent_runtime(
        agentRuntimeId=runtime_id,
        networkConfiguration={
            "networkMode": "VPC",
            "networkModeConfig": {
                "subnets": ["subnet-xxxxxxxxxxxxxxxxx", "subnet-yyyyyyyyyyyyyyyyy"],
                "securityGroups": ["sg-xxxxxxxxxxxxxxxxx"],
            },
        },
        agentRuntimeArtifact={
            "containerConfiguration": {
                "containerUri": <CONTAINER_URI>  # TODO: Set your container URI, e.g., "123456789012.dkr.ecr.ap-southeast-2.amazonaws.com/firewall-automation/firewall-logs-agent:latest"
            }
        },
        roleArn="arn:aws:iam::123456789012:role/YourAgentCoreExecutionRole",
    )
    
print(f"Firewall Logs Agent deployed successfully!")

## Step 6: Test End-to-End

Test the deployed agent through the Streamlit UI.

## Step 7: Push Your Changes

Create a branch and push your changes for review.

## Tips and Best Practices

### Security
- Store OpenSearch endpoint and role ARN in environment variables or AWS Secrets Manager
- Use IAM roles for authentication, not access keys
- Implement proper error handling to avoid leaking sensitive information

### Performance
- Add caching with 5-minute TTL to reduce OpenSearch load
- Set reasonable default time ranges (1 hour)
- Limit result sets to prevent memory issues
- Use OpenSearch aggregations for summaries instead of processing large result sets

### User Experience
- Provide clear error messages when queries fail
- Show progress indicators for long-running queries
- Format results in a readable way
- Highlight security-relevant information (blocked traffic, suspicious patterns)

## Resources

- Example: `network_firewall_analyser_agent.py`
- README: `README.md` in this folder
- Jira: [FWAUTO-13](https://your-jira-instance.atlassian.net/browse/FWAUTO-13)
- OpenSearch Query DSL Documentation